# Ube Craze Comments Preprocessing

This notebook processes the raw JSON scraped comments from `data/raw/` by applying normalization, tokenization, language detection (using `lingua-language-detector`), and filtering out non-target commentary (keeping only English, Tagalog, and Taglish).

## 1. Imports and Config

In [ ]:
import json
import pandas as pd
from pathlib import Path
from ube_craze_nlp.nlp.clean import normalize_text, is_target_language, detect_language
from ube_craze_nlp.scraper.parser import parse_comments_payload
from ube_craze_nlp.utils.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR, ensure_dirs

ensure_dirs()
print(f"Reading raw from: {RAW_DATA_DIR}")
print(f"Saving processed to: {PROCESSED_DATA_DIR}")

## 2. Load Raw Comments into a DataFrame

In [ ]:
raw_directories = [d for d in RAW_DATA_DIR.iterdir() if d.is_dir()]
all_comments = []

for r_dir in raw_directories:
    raw_file = r_dir / "comments_raw.json"
    metadata_file = r_dir / "metadata.json"

    if not raw_file.exists():
        continue

    # Load video metadata
    with open(metadata_file, "r", encoding="utf-8") as f:
        meta = json.load(f)

    # Load and parse comments payloads
    with open(raw_file, "r", encoding="utf-8") as f:
        payloads = json.load(f)

    # Parse comment objects
    for payload in payloads:
        parsed_list = parse_comments_payload(payload)
        for comment in parsed_list:
            # Add video-level identifiers
            comment["video_id"] = meta.get("video_id")
            comment["video_author"] = meta.get("author")
            all_comments.append(comment)

df_raw = pd.DataFrame(all_comments)
print(f"Loaded {len(df_raw)} total comments from {len(raw_directories)} video directories.")
df_raw.head()

## 3. Preprocess and Clean Text

We clean URLs, usernames, and excessive spaces. We then detect the language and filter out comments that are not in English or Tagalog/Taglish (e.g. Spanish, Indonesian, Vietnamese, Arabic).

In [ ]:
if df_raw.empty:
    print("No raw comments found. Run the scraper first!")
else:
    # Remove duplicates based on unique comment ID
    df_clean = df_raw.drop_duplicates(subset=["comment_id"]).copy()
    print(f"Unique comments count: {len(df_clean)} (Removed {len(df_raw) - len(df_clean)} duplicate items)")

    # Normalize text for better language identification
    df_clean["normalized_text"] = df_clean["text"].apply(normalize_text)

    # Filter out empty or whitespace-only comments
    df_clean = df_clean[df_clean["normalized_text"].str.strip() != ""].copy()

    # Detect language using Lingua
    print("⏳ Running language detection on unique comments...")
    df_clean["detected_language"] = df_clean["normalized_text"].apply(detect_language)

    # Keep English, Tagalog (which captures Taglish), and undefined (None)
    df_filtered = df_clean[df_clean["detected_language"].isin(["en", "tl", None])].copy()

    removed_count = len(df_clean) - len(df_filtered)
    print(f"Filtered out {removed_count} foreign language comments.")
    print(f"Remaining target comments: {len(df_filtered)}")

    # Language distribution breakdown
    print("\nLanguage Breakdown:")
    print(df_filtered["detected_language"].value_counts(dropna=False))

## 4. Export Cleaned Dataset

In [ ]:
if not df_raw.empty:
    processed_file = PROCESSED_DATA_DIR / "cleaned_comments.csv"
    df_filtered.to_csv(processed_file, index=False, encoding="utf-8-sig")
    print(f"✅ Preprocessed dataset saved successfully to: {processed_file}")